In [4]:
from pathlib import Path
import pymupdf

PDF_PATH = next(Path("pdfs").glob("*.pdf"))   # prend le PDF du dossier pdfs
print("PDF utilisé :", PDF_PATH)

OUTPUT_DIR = Path("resultats_pdf")
TEXT_DIR, IMAGE_DIR = OUTPUT_DIR / "textes", OUTPUT_DIR / "images"
TEXT_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

textes = []
with pymupdf.open(PDF_PATH) as doc:
    for num, page in enumerate(doc, start=1):
        texte = page.get_text("text", sort=True)
        (TEXT_DIR / f"page_{num:03d}.txt").write_text(texte, encoding="utf-8")
        textes.append(f"===== PAGE {num} =====\n\n{texte}")
        vues = set()
        for info in page.get_images(full=True):
            xref = info[0]
            if xref in vues:
                continue
            vues.add(xref)
            img = doc.extract_image(xref)
            if img:
                (IMAGE_DIR / f"page_{num:03d}_image_{xref}.{img['ext']}").write_bytes(img["image"])

(OUTPUT_DIR / "texte_complet.txt").write_text("\n\n".join(textes), encoding="utf-8")
print(len(list(TEXT_DIR.iterdir())), "pages,", len(list(IMAGE_DIR.iterdir())), "images")

PDF utilisé : pdfs\attention is all you need.pdf
14 pages, 3 images


In [5]:
import torch, open_clip
from PIL import Image
from open_clip.tokenizer import SimpleTokenizer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME, PRETRAINED = "ViT-B-32", "laion2b_s34b_b79k"
model, _, preprocess = open_clip.create_model_and_transforms(MODEL_NAME, pretrained=PRETRAINED)
model = model.to(DEVICE).eval()
tokenizer = open_clip.get_tokenizer(MODEL_NAME)
bpe = SimpleTokenizer()

@torch.inference_mode()
def encoder_textes(textes):
    v = model.encode_text(tokenizer(textes).to(DEVICE)).float()
    return (v / v.norm(dim=-1, keepdim=True).clamp_min(1e-12)).cpu().tolist()

@torch.inference_mode()
def encoder_image(chemin):
    with Image.open(chemin) as im:
        t = preprocess(im.convert("RGB")).unsqueeze(0).to(DEVICE)
    v = model.encode_image(t).float()
    return (v / v.norm(dim=-1, keepdim=True).clamp_min(1e-12)).cpu().tolist()[0]

def decouper_texte(texte, taille=60, chevauchement=10):
    tokens = bpe.encode(texte)
    for d in range(0, len(tokens), taille - chevauchement):
        frag = bpe.decode(tokens[d:d + taille]).strip()
        if frag:
            yield frag
        if d + taille >= len(tokens):
            break

In [6]:
v_texte = encoder_textes(["a diagram of the transformer architecture"])[0]
print("Dimension :", len(v_texte))

Dimension : 512


In [7]:
import chromadb, hashlib, re

client = chromadb.PersistentClient(path="chroma_pdf")
collection = client.get_or_create_collection(
    "pdf_clip_vit_b32", embedding_function=None, metadata={"hnsw:space": "cosine"})

def numero_page(p):
    m = re.search(r"page_(\d+)", Path(p).stem)
    return int(m.group(1)) if m else -1

# Identifiant lié au contenu du PDF (permet de réindexer proprement)
document_id = hashlib.sha256(PDF_PATH.read_bytes()).hexdigest()[:24]
collection.delete(where={"document_id": document_id})

# --- Textes ---
ids, docs, metas = [], [], []
for f in sorted(TEXT_DIR.glob("page_*.txt")):
    page = numero_page(f)
    for i, frag in enumerate(decouper_texte(f.read_text(encoding="utf-8"))):
        ids.append(f"{document_id}:texte:{page}:{i}")
        docs.append(frag)
        metas.append({"type": "texte", "document_id": document_id,
                      "pdf": PDF_PATH.name, "page": page, "fragment": i})

for d in range(0, len(docs), 64):
    collection.upsert(ids=ids[d:d+64], documents=docs[d:d+64],
                      metadatas=metas[d:d+64],
                      embeddings=encoder_textes(docs[d:d+64]))
print(len(docs), "fragments de texte indexés")

# --- Images ---
nb_images = 0
for f in sorted(IMAGE_DIR.iterdir()):
    if f.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp", ".bmp"}:
        collection.upsert(
            ids=[f"{document_id}:image:{f.name}"],
            embeddings=[encoder_image(f)],
            documents=[f"Image extraite du PDF : {f.name}"],
            metadatas=[{"type": "image", "document_id": document_id,
                        "pdf": PDF_PATH.name, "page": numero_page(f),
                        "source": str(f.resolve())}])
        nb_images += 1
print(nb_images, "images indexées")
print("Total dans ChromaDB :", collection.count())

172 fragments de texte indexés
3 images indexées
Total dans ChromaDB : 175
